In [3]:
# !pip install -r requirements.txt

In [7]:
# !pip install --upgrade pip
# !pip install ipywidgets
# !pip install jupyterlab_widgets

In [8]:
import pandas as pd
from tqdm import tqdm
import json
from itertools import islice

from river import metrics, dummy, forest, tree, stats, ensemble, drift
from ensemble import DriftAdaptiveEnsemble
from transformer import FeatureDistrict
from utils import warsaw_stream, airline_stream, taxi_stream

seed = 17

with open("transformer/config_dataset.json", "r", encoding="utf-8-sig") as f:
    config = json.load(f)

In [9]:
DATASET = 'Airplanes'
MAX_SAMPLES = 500_000

In [11]:
# df_warsaw=pd.read_csv("dataset/warsaw_synthetic.csv")
df_airplanes=pd.read_csv("dataset/Airplanes_modified.csv")
# df_taxi=pd.read_csv("dataset/taxi_dataset_ordered.csv")

if DATASET=="Airplanes":
    rows = df_airplanes.to_dict(orient='records')
    data = airline_stream(rows)
elif DATASET=="Warsaw":
    rows = df_warsaw.to_dict(orient='records')
    data = warsaw_stream(rows)
elif DATASET=="Taxi":
    rows = df_taxi.to_dict(orient='records')
    data = taxi_stream(rows)
else:
    raise Exception("Dataset ERROR")

cfg = config[DATASET]

In [13]:
models = {
    'MEAN': dummy.StatisticRegressor(stats.Mean()),
    'HTR': tree.HoeffdingTreeRegressor(),
    "ARF": forest.ARFRegressor(n_models=10, seed=seed, max_depth=30, drift_detector=None, warning_detector=None),
    "ARF_drift": forest.ARFRegressor(n_models=10, seed=seed, drift_detector=drift.ADWIN(0.001), warning_detector=drift.ADWIN(0.01), max_depth=30),
    "SRP": ensemble.SRPRegressor(n_models=10, seed=seed, model=tree.HoeffdingTreeRegressor(max_depth=30), drift_detector=None, warning_detector=None),
    "SRP_drift": ensemble.SRPRegressor(n_models=10, seed=seed, drift_detector=drift.ADWIN(0.001), warning_detector=drift.ADWIN(0.01), model=tree.HoeffdingTreeRegressor(max_depth=30)),
    "Ensemble": DriftAdaptiveEnsemble(
            base_estimator=tree.HoeffdingTreeRegressor(max_depth=30),
            drift_detector=drift.ADWIN(0.001), warning_detector=drift.ADWIN(0.01),
            metric=metrics.RMSE(),
            max_ensemble_size=10, 
            retain_initial_model=False
        )
}
transformer = FeatureDistrict(
    dataset=DATASET,
    columns_to_drop=cfg["columns_to_drop"],
)

In [16]:
data = islice(data, MAX_SAMPLES)

In [17]:
x, y = next(data)
print(x)
print(y)

{'Month': 1, 'DayofMonth': 1, 'DayOfWeek': 2, 'DepTime': 5.0, 'CRSDepTime': 15, 'CRSArrTime': 823, 'CRSElapsedTime': 308.0, 'Distance': 2521, 'previous_dep_delay': 7.0, 'scheduled_turnaround': 980.0, 'origin_state': 'CA', 'origin_lat': 38.69542167, 'origin_long': -121.5907669, 'dest_state': 'NY', 'dest_lat': 40.63975111, 'dest_long': -73.77892556}
-19.0


In [18]:
N = len(rows)
print(N)

2319121


In [19]:
rows = []
for i, (x_raw, y) in enumerate(tqdm(data, total=MAX_SAMPLES)):
    x = transformer.transform_one(x_raw)
    timestamp = x['timestamp']
    x.pop("timestamp")

    x.pop('pickup_district', None)
    x.pop('dropoff_district', None)
    x.pop('within_district', None)

    # print(x)

    row = {
        "i": i,
        "timestamp": timestamp,
        "y_true": y,
    }

    for name, model in models.items():
        pred = model.predict_one(x)
        row[f"y_{name}"] = pred if pred is not None else 0.0
        model.learn_one(x, y)

    rows.append(row)

 51%|█████     | 253094/500000 [3:57:45<3:51:56, 17.74it/s]     


KeyboardInterrupt: 

In [ ]:
pd.DataFrame(rows).to_csv('results/compare_models/airline.csv', index=None)

In [37]:
len(pd.DataFrame(rows))

298252